# Comparar modelos e validação cruzada temporal

Como ler o resultado da busca e comparar modelos com rigor:

1. **Ranking das finalistas** na camada de seleção (`V_select`).
2. **Validação cruzada temporal** (rolling origin) com dobras próprias, para comparar vários modelos nas mesmas origens.
3. **Testes estatísticos** (Wilcoxon e Diebold-Mariano com Holm; Friedman com diferença crítica de Nemenyi).
4. **Comparação final no teste selado**, uma avaliação por modelo.
5. **Validação cruzada aninhada** do procedimento inteiro: quanto a seleção automática generaliza.

Tabelas e testes servem ao **relatório**: a escolha do campeão segue a regra do protocolo, e o teste selado nunca volta para a busca.

In [ ]:
import autohybts

print("AutoHybTS", autohybts.__version__)

## 1. Experimento e busca

In [ ]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
exp = AutoHybridExperiment(
    fh=14,
    session_id=42,
    fold=3,
    base_models=["naive", "seasonal_naive", "linear", "knn", "mlp"],
    correctors=["linear", "knn"],
    residual_windows=[7],
).fit(y)
result = exp.compare_models()
result.best

## 2. Ranking das finalistas em V_select

Uma finalista por família, com o campeão como referência (primeira linha da comparação). `mean_rank` é o posto médio entre as origens de seleção.

In [ ]:
result.comparison.summary

In [ ]:
result.comparison.plot("leaderboard")

## 3. Validação cruzada temporal

Dobras próprias sobre o histórico de desenvolvimento (o teste selado fica de fora): 8 origens, uma a cada 7 dias, cada uma prevendo 14 passos. Todos os modelos são avaliados **nas mesmas origens**. Opções: `folds`, `step`, `fold_strategy` (`"expanding"` ou `"rolling"`), `window_size` e `horizon`.

In [ ]:
cv = exp.cross_validate(folds=8, step=7)
cv.splits

In [ ]:
cv.summary

In [ ]:
cv.units  # MASE por origem e modelo

### Visualizações

In [ ]:
cv.plot("leaderboard")

In [ ]:
cv.plot("boxplot")

In [ ]:
cv.plot("metric_by_fold")

In [ ]:
cv.plot("win_rate")

In [ ]:
cv.plot("forecasts")

### Testes estatísticos

Cada modelo contra a referência (o campeão): Wilcoxon pareado e Diebold-Mariano (variância de Newey-West, para origens com alvos sobrepostos), com correção de Holm. O teste de Friedman e a diferença crítica de Nemenyi comparam todos ao mesmo tempo.

Com poucas origens os testes têm pouco poder: "sem diferença significativa" não quer dizer que os modelos são equivalentes.

In [ ]:
cv.tests()

In [ ]:
friedman = cv.friedman()
p_value, cd = friedman["p_value"], friedman["critical_difference"]
print(f"Friedman p = {p_value:.3f}; diferença crítica = {cd:.2f}")
friedman["mean_ranks"].sort_values()

In [ ]:
cv.plot("critical_difference")

## 4. Comparação final no teste selado

Cada modelo é finalizado (reajuste no histórico anterior ao teste) e avaliado **uma única vez**. É um relato terminal: mudar a configuração por causa dele transformaria o teste em validação.

In [ ]:
external = exp.compare_external()
external.table

In [ ]:
external.tests

## 5. Validação cruzada aninhada do procedimento

Em cada dobra externa, a AutoHybTS refaz busca e seleção usando só o passado e é medida no bloco seguinte, ao lado de referências fixas. Responde à pergunta: *escolher automaticamente compensa em relação a usar sempre o naive sazonal?*

In [ ]:
nested = exp.nested_cross_validate(outer_folds=3, budget="smoke")
nested.folds[["champion", "strategy", "autohybts_MASE", "naive_MASE", "seasonal_naive_MASE"]]

In [ ]:
nested.summary